In [1]:
import polars as pl
import numpy as np
from collections import defaultdict
import os

merged_final = pl.read_parquet("../data/merged_final_2.parquet")
domain_to_idx = {domain: idx for idx, domain in enumerate(merged_final["domain"].to_list())}
valid_domains = set(domain_to_idx.keys())

edge_files = ["../data/edge_edges.parquet", "../data/merged_edges.parquet"]
all_edges = []

for file in edge_files:
    if os.path.exists(file):
        df = pl.read_parquet(file)
        all_edges.append(df)

if not all_edges:
    raise FileNotFoundError("No edge files found")

edges_df = pl.concat(all_edges)
edges_df = edges_df.filter((pl.col("source").is_in(valid_domains)) & (pl.col("target").is_in(valid_domains)))
edges_df = edges_df.unique()

source_neighbors = defaultdict(list)
for row in edges_df.iter_rows(named=True):
    source_neighbors[row["source"]].append(row["target"])

count_embeddings = np.load("../weights/count_vectorizer_embeddings.npy")
embedding_dim = count_embeddings.shape[1]

neighbor_embeddings = np.zeros((len(merged_final), embedding_dim), dtype=np.float32)

for domain, idx in domain_to_idx.items():
    neighbors = source_neighbors.get(domain, [])
    if not neighbors:
        continue
    
    neighbor_indices = []
    for neighbor in neighbors:
        if neighbor in domain_to_idx:
            neighbor_indices.append(domain_to_idx[neighbor])
    
    if not neighbor_indices:
        continue
    
    if len(neighbor_indices) > 5:
        neighbor_indices = neighbor_indices[:5]
    
    neighbor_vectors = count_embeddings[neighbor_indices]
    neighbor_embeddings[idx] = np.mean(neighbor_vectors, axis=0)

np.save("../weights/neighbor_embeddings.npy", neighbor_embeddings)